In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

In [23]:
ticker = "^DJI"
df_price = yf.download(ticker, start="2008-01-01", end="2016-12-31")

df_price = df_price[["Close"]]
df_price["return"] = df_price["Close"].pct_change()
df_price["target"] = (df_price["return"].shift(-1) > 0).astype(int)

df_price = df_price.dropna()
df_price.head()

# Adding lagged and rolling average features
for lag in [1, 2, 3, 5]:
    df_price[f"lag_{lag}"] = df_price["return"].shift(lag)

df_price["roll_mean_5"] = df_price["return"].rolling(5).mean()
df_price["roll_std_5"] = df_price["return"].rolling(5).std()

df_price = df_price.dropna()

/tmp/ipykernel_1055/2645146144.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_price = yf.download(ticker, start="2008-01-01", end="2016-12-31")
[*********************100%***********************]  1 of 1 completed


In [24]:
df_news = pd.read_csv(
    "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Combined_News_DJIA.csv"
)

df_news.columns = df_news.columns.str.strip()
df_news["Date"] = pd.to_datetime(df_news["Date"], dayfirst=True)

headline_cols = [c for c in df_news.columns if c.startswith("Top")]

df_news[headline_cols] = df_news[headline_cols].fillna("")

df_news["combined"] = df_news[headline_cols].apply(
    lambda row: " ".join([str(x) for x in row if str(x).strip() != ""]),
    axis=1
)

df_news["combined"].head()

,combined
0,"b""Georgia 'downs two Russian warplanes' as cou..."
1,b'Why wont America and Nato help us? If they w...
2,b'Remember that adorable 9-year-old who sang a...
3,b' U.S. refuses Israel weapons to attack Iran:...
4,b'All the experts admit that we should legalis...


In [25]:
import ast

def clean_bytes_text(x):
    """
    Convert byte-like strings (b'...') to normal text
    """
    if isinstance(x, str):
        try:
            evaluated_value = ast.literal_eval(x)
            if isinstance(evaluated_value, bytes):
                return evaluated_value.decode('utf-8')
            else:
                return str(evaluated_value)
        except (ValueError, SyntaxError):
            if x.startswith("b"):
                return x.strip("b'\"")
            return x
    return x

df_news["combined"] = df_news["combined"].apply(clean_bytes_text)

In [26]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_news["combined"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [27]:
pca = PCA(n_components=20)
embeddings_reduced = pca.fit_transform(embeddings)

emb_df = pd.DataFrame(
    embeddings_reduced,
    index=df_news["Date"],
    columns=[f"emb_{i}" for i in range(20)]
)

In [28]:
emb_df = emb_df.copy()
emb_df.reset_index(inplace=True)
emb_df.rename(columns={emb_df.columns[0]: "Date"}, inplace=True)

if 'level_0' in df_price.columns:
    df_price = df_price.drop(columns=['level_0'])
if 'index' in df_price.columns:
    df_price = df_price.drop(columns=['index'])

df_price_cleaned = df_price.reset_index()

if isinstance(df_price_cleaned.columns, pd.MultiIndex):
    new_cols = []
    for col in df_price_cleaned.columns:
        if col[0] == 'Date':
            new_cols.append('Date')
        elif isinstance(col, tuple) and col[1] != '':
            new_cols.append(col[1])
        else:
            new_cols.append(col[0])
    df_price_cleaned.columns = new_cols

if 'Ticker' in df_price_cleaned.columns:
    df_price_cleaned = df_price_cleaned.drop(columns=['Ticker'])

df_price_cleaned['Date'] = pd.to_datetime(df_price_cleaned['Date'])

df = pd.merge(df_price_cleaned, emb_df, on="Date", how="inner")

In [29]:
emb_features = [col for col in df.columns if "emb_" in col]

ts_features = [col for col in df.columns if "lag_" in col or "roll_" in col]

In [30]:
# IMPORTANT: set 'Date' as the DataFrame index for proper time series splitting
df = df.set_index('Date')

train = df[df.index < "2014-01-01"]  # Splitting threshold
test  = df[df.index >= "2014-01-01"]

X_train_base = train[ts_features]
X_test_base  = test[ts_features]

X_train_full = train[ts_features + emb_features]
X_test_full  = test[ts_features + emb_features]

y_train = train["target"]
y_test  = test["target"]

In [31]:
# Baseline model: no LLM embeddings
model_base = LGBMClassifier(random_state=42)
model_base.fit(X_train_base, y_train)

pred_base = model_base.predict(X_test_base)
acc_base = accuracy_score(y_test, pred_base)

print("Baseline Accuracy:", acc_base)

[LightGBM] [Info] Number of positive: 734, number of negative: 625
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000122 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 1359, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.540103 -> initscore=0.160757
[LightGBM] [Info] Start training from score 0.160757
Baseline Accuracy: 0.5


In [32]:
# Full model with LLM embeddings
model_full = LGBMClassifier(random_state=42)
model_full.fit(X_train_full, y_train)

pred_full = model_full.predict(X_test_full)
acc_full = accuracy_score(y_test, pred_full)

print("With Embeddings Accuracy:", acc_full)

[LightGBM] [Info] Number of positive: 734, number of negative: 625
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000400 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6630
[LightGBM] [Info] Number of data points in the train set: 1359, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.540103 -> initscore=0.160757
[LightGBM] [Info] Start training from score 0.160757
With Embeddings Accuracy: 0.5238095238095238
